# Surface Defect Inspection (UC1)

Simulate a scratched surface under directional illumination, detect the scratch via contrast analysis, and make a pass/fail decision.

In [ ]:
import numpy as np
from optical_metrology.illumination import Laser, GaussianBeamProfile
from optical_metrology.surface import ScratchedSurface, Material
from optical_metrology.scattering import LambertianScattering
from optical_metrology.optics import OpticalSystem, GaussianPSF, OpticalPropagator
from optical_metrology.detector import CMOSDetector
from optical_metrology.analysis import ContrastAnalyzer, SaturationAnalyzer, ImageAnalyzer

In [ ]:
shape = (32, 32)

laser = Laser(wavelength=532e-9, power=5e-3, beam_profile=GaussianBeamProfile(w0=3.0))
laser.propagation_direction = np.array([0.0, 0.0, -1.0])

surface = ScratchedSurface(shape, scratch_depth=0.3, scratch_width=3, material=Material("silicon"))
print(f"Scratch depth: {surface.height.min():.3g}, roughness: {surface.roughness:.4g}")

In [ ]:
lf = laser.generate_light_field(shape=shape, spacing=0.5)
scattered = LambertianScattering(albedo=0.7).evaluate(lf, surface, view_direction=[0, 0, 1])
optics = OpticalSystem(focal_length=0.05, aperture_diameter=0.008, wavelength=532e-9)
sensor = OpticalPropagator(GaussianPSF(sigma=1.0)).propagate(scattered, optics)
image = CMOSDetector(exposure_time=1e-5, gain=1.0).capture(sensor)

In [ ]:
analyzer = ImageAnalyzer(modules=[ContrastAnalyzer(), SaturationAnalyzer()])
report = analyzer.analyze(image)
for k, v in report.measurements.items():
    print(f"  {k}: {v:.4g}")

In [ ]:
# Simple pass/fail: flag if contrast exceeds a threshold
contrast = report.measurements.get("rms_contrast", 0)
threshold = 0.05
if contrast > threshold:
    print(f"FAIL — contrast {contrast:.4f} exceeds threshold {threshold}")
else:
    print(f"PASS — contrast {contrast:.4f} within tolerance")

In [ ]:
print(image.visualize(max_width=48))